# Problem: Implement a Deep Neural Network

### Problem Statement
You are tasked with constructing a **Deep Neural Network (DNN)** model to solve a regression task using PyTorch. The objective is to predict target values from synthetic data exhibiting a non-linear relationship.

### Requirements
Implement the `DNNModel` class that satisfies the following criteria:

1. **Model Definition**:
   - The model should have:
     - An **input layer** connected to a **hidden layer** (input: 2 features).
     - A **ReLU activation function** for non-linearity.
     - An **output layer** with a single unit for regression.
   - `forward` maps a batch of shape `(N, 2)` to predictions of shape `(N, 1)`.

The target here is $y = \tfrac{1}{2}(x_1 - 5)^2 + x_2 + \varepsilon$ — quadratic in the first feature, so no linear model can fit it. The last cell trains a plain `nn.Linear(2, 1)` on the same data for comparison.

<details> <summary>💡 Hint</summary> - Use `nn.Sequential` to simplify the implementation of the `DNNModel`. - Experiment with different numbers of layers and hidden units to optimize performance. - Ensure the final layer has a single output unit (since it's a regression task). </details> <details> <summary>💡 Bonus: Try Custom Loss Functions</summary> Experiment with custom loss functions (e.g., Huber Loss) and compare their performance with MSE. </details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.rand(100, 2) * 10  # 100 data points with 2 features
y = (0.5 * (X[:, 0] - 5) ** 2 + X[:, 1]).unsqueeze(1) + torch.randn(100, 1)  # Non-linear (quadratic in x1) with noise

In [ ]:
# Define the Deep Neural Network Model
class DNNModel(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden_dim)  # Input layer to hidden layer
        self.relu = nn.ReLU()                # Non-linearity: without it fc2(fc1(x)) is still linear
        self.fc2 = nn.Linear(hidden_dim, 1)  # Hidden layer to output layer

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
# Initialize the model, loss function, and optimizer
model = DNNModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

# Testing on new data (noise-free targets: 0.5 * (x1 - 5)^2 + x2 -> [3.5, 10.0])
X_test = torch.tensor([[4.0, 3.0], [7.0, 8.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

In [ ]:
# Does the hidden layer + ReLU matter? Fit a purely linear model to the same data.
linear_model = nn.Linear(2, 1)
linear_optimizer = optim.Adam(linear_model.parameters(), lr=0.01)
for epoch in range(epochs):
    linear_optimizer.zero_grad()
    linear_loss = criterion(linear_model(X), y)
    linear_loss.backward()
    linear_optimizer.step()

print(f"Final training MSE - DNN: {loss.item():.4f}, linear model: {linear_loss.item():.4f}")
print("(the noise variance is 1.0: that is the best achievable MSE on new data; a training MSE\n"
      " well below it means the network has started to fit the noise)")